# 03 · Silver del Desempeño Municipal (MDM)
**Responsable:** P1 · **Código:** `src/silver_desempeno.py` · **Salida:** `silver/desempeno.parquet`

**Qué hace**
- El archivo trae **246 filas para 123 municipios**: dos bloques con valores distintos, ambos rotulados 2020. Se marca cada fila con `bloque` A (filas 1–123) o B (124–246) y se conservan los dos.
- Compara el nombre de cada municipio con el nombre DANE de su código, sin tildes. En el bloque A, 15022 y 15047 tienen los nombres cruzados (Aquitania / Almeida).
- `es_bloque_oficial` marca el bloque elegido en `config.MDM_BLOQUE_OFICIAL`. En ese bloque, un código con nombre de otro municipio se rechaza; en el otro solo se alerta.
- Valida puntajes entre 0 y 100 y el grupo de capacidades iniciales (G1–G6).

**Decisión pendiente del equipo:** confirmar en el portal cuál bloque corresponde a 2020 y dejarlo en `config.py`. El otro se usa en la sensibilidad.

**Qué revisar al ejecutar:** `desempeno_nombres_incoherentes.csv` y `desempeno_comparacion_bloques.csv` en `outputs/quality/`.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import polars as pl
from src.config import *
asegurar_carpetas()

In [2]:
from src.silver_desempeno import construir_desempeno
pob = pl.read_parquet(SILVER / "poblacion.parquet")
mdm = construir_desempeno(pob)
mdm.group_by("bloque", "es_bloque_oficial").len()

[silver/desempeno] revisados=246 validos=246 rechazados=0


bloque,es_bloque_oficial,len
str,bool,u32
"""B""",true,123
"""A""",false,123


In [3]:
pl.read_csv(QUALITY / "desempeno_nombres_incoherentes.csv")

bloque,fila,codigo_municipio,municipio_fuente,municipio_dane,puntaje_mdm
str,i64,i64,str,str,f64
"""A""",1,15022,"""Aquitania""","""Almeida""",55.73
"""A""",2,15047,"""Almeida""","""Aquitania""",52.08


In [4]:
pl.read_csv(QUALITY / "desempeno_comparacion_bloques.csv").describe()

statistic,codigo_municipio,mdm_a,mdm_b,diferencia
str,f64,f64,f64,f64
"""count""",123.0,123.0,123.0,123.0
"""null_count""",0.0,0.0,0.0,0.0
"""mean""",15481.699187,52.698374,54.939106,2.240732
"""std""",254.524563,7.578651,7.917878,8.415188
"""min""",15001.0,34.4,33.2,-18.96
"""25%""",15238.0,48.06,49.56,-2.54
"""50%""",15500.0,52.08,55.64,2.35
"""75%""",15723.0,57.0,61.19,8.26
"""max""",15897.0,73.0,69.97,20.62
